In [1]:
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr


from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os



embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db_name = "../vector_database"

vector_database = Chroma(persist_directory=db_name, embedding_function=embeddings)

c:\Users\ASUS\Desktop\scientific-paper-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10091.18it/s]


In [2]:
AZURE_ENDPOINT = (
    "https://ankitsinghtheweeknd691-9348-reso.services.ai.azure.com/openai/v1"
)

DEFAULT_MODEL = "Mistral-Large-3"

In [ ]:

llm = ChatOpenAI(
    base_url=AZURE_ENDPOINT,
    api_key=os.getenv("AZURE_FOUNDRY_API_KEY"),
    model=DEFAULT_MODEL,
    default_query={"api-version": "preview"},
)

retriever = vector_database.as_retriever(search_kwargs={"k": 5})

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant answering questions about Insurellm. "
            "Use only the provided context to answer. If the answer isn't in "
            "the context, say you don't know.\n\n"
            "Context:\n{context}",
        ),
        ("human", "Question: {question}"),
    ]
)


def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('doc_type', 'unknown')} | {d.metadata.get('source', 'unknown')}]\n{d.page_content}"
        for d in docs
    )


"""
RAG LCEL CHAIN
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Who is the CEO of Insurellm?")
print(answer)
"""


def run_rag(question: str) -> str:
    docs = retriever.invoke(question)

    context = format_docs(docs)

    messages = prompt.invoke({"context": context, "question": question})

    response = llm.invoke(messages)

    answer = StrOutputParser().invoke(response)

    return answer


answer = run_rag("What is Insurellm?")
print(answer)


In [ ]:
def chat(message, history):
    return run_rag(message)

demo = gr.ChatInterface(
    fn=chat,
    title="Insurellm Knowledge Base Assistant",
    
)
demo.launch()
